In [13]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    regression_dir: Path
    ensemble_dir: Path
    lstm_dir: Path
    scaler_name: str
    lstm_scaler_name: str
    target_column: str


In [3]:
import os
os.chdir("..")
%pwd

'd:\\Projects\\RAG_Systems\\ProductDemand\\product-demand-forecasting-system'

In [14]:
import sys
from pathlib import Path
from src.productdemand.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.productdemand.utils.common import read_yaml, create_directories
from src.productdemand.exception.custom_exception import CustomException

class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE_PATH,
            schema_filepath=SCHEMA_FILE_PATH,
        ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([Path(self.config.artifacts_root)])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        schema = self.schema.TARGET_COLUMN

        create_directories([
            Path(config.root_dir),
            Path(config.regression_dir),
            Path(config.ensemble_dir),
            Path(config.lstm_dir)
        ])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            regression_dir=Path(config.regression_dir),
            ensemble_dir=Path(config.ensemble_dir),
            lstm_dir=Path(config.lstm_dir),
            scaler_name=config.scaler_name,
            lstm_scaler_name=config.lstm_scaler_name,
            target_column=schema.name
        )

        return data_transformation_config


In [15]:
import os, sys
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler

from src.productdemand.exception.custom_exception import CustomException

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def clean_target_column(self, df: pd.DataFrame) -> pd.DataFrame:
        target = self.config.target_column

        if target not in df.columns:
            raise ValueError(f"Target column '{target}' not found in dataframe")

        df[target] = df[target].astype(str)

        # اگر مقدارها مثل (100) باشند، یعنی مقدار منفی
        df[target] = df[target].str.replace("(", "-", regex=False)
        df[target] = df[target].str.replace(")", "", regex=False)

        # حذف فاصله و کاما
        df[target] = df[target].str.replace(",", "", regex=False)
        df[target] = df[target].str.strip()

        df[target] = pd.to_numeric(df[target], errors="coerce")

        return df

    def feature_engineering(self, df: pd.DataFrame) -> pd.DataFrame:
        if "Date" not in df.columns:
            raise ValueError("Date column not found in dataframe")

        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

        df = df.dropna(subset=["Date"])

        df = df.sort_values("Date").reset_index(drop=True)

        df["year"] = df["Date"].dt.year
        df["month"] = df["Date"].dt.month
        df["day"] = df["Date"].dt.day
        df["dayofweek"] = df["Date"].dt.dayofweek
        df["weekofyear"] = df["Date"].dt.isocalendar().week.astype(int)

        return df

    def handle_missing_values(self, df: pd.DataFrame) -> pd.DataFrame:
        target = self.config.target_column

        # حذف ردیف‌هایی که target ندارند
        df = df.dropna(subset=[target])

        # پر کردن عددی‌ها با median
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            df[col] = df[col].fillna(df[col].median())

        # پر کردن objectها با mode
        object_cols = df.select_dtypes(include=["object"]).columns
        for col in object_cols:
            if df[col].mode().empty:
                df[col] = df[col].fillna("Unknown")
            else:
                df[col] = df[col].fillna(df[col].mode()[0])

        return df

    def handle_encoding(self, df: pd.DataFrame) -> pd.DataFrame:
        object_cols = df.select_dtypes(include=["object"]).columns

        label_encoders = {}

        for col in object_cols:
            encoder = LabelEncoder()
            df[col] = df[col].astype(str)
            df[col] = encoder.fit_transform(df[col])
            label_encoders[col] = encoder

        encoder_path = os.path.join(self.config.root_dir, "label_encoders.pkl")
        joblib.dump(label_encoders, encoder_path)

        return df

    def create_sequences(self, data: np.ndarray, window_size: int = 30):
        X, y = [], []

        for i in range(len(data) - window_size):
            X.append(data[i:i + window_size, :-1])
            y.append(data[i + window_size, -1])

        return np.array(X), np.array(y)

    def save_regression_data(self, train: pd.DataFrame, test: pd.DataFrame):
        target = self.config.target_column

        X_train = train.drop(columns=[target])
        y_train = train[target]

        X_test = test.drop(columns=[target])
        y_test = test[target]

        # اطمینان از اینکه فقط عددی‌ها وارد scaler شوند
        X_train = X_train.select_dtypes(include=[np.number])
        X_test = X_test[X_train.columns]

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        joblib.dump(
            scaler,
            os.path.join(self.config.root_dir, self.config.scaler_name)
        )

        train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
        train_scaled[target] = y_train.values

        test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)
        test_scaled[target] = y_test.values

        train_scaled.to_csv(
            os.path.join(self.config.regression_dir, "train.csv"),
            index=False
        )

        test_scaled.to_csv(
            os.path.join(self.config.regression_dir, "test.csv"),
            index=False
        )

    def save_ensemble_data(self, train: pd.DataFrame, test: pd.DataFrame):
        train.to_csv(
            os.path.join(self.config.ensemble_dir, "train.csv"),
            index=False
        )

        test.to_csv(
            os.path.join(self.config.ensemble_dir, "test.csv"),
            index=False
        )

    def save_lstm_data(self, df: pd.DataFrame, window_size: int = 30):
        target = self.config.target_column

        lstm_df = df.copy()

        if "Date" in lstm_df.columns:
            lstm_df = lstm_df.drop(columns=["Date"])

        # target را آخرین ستون قرار می‌دهیم
        feature_cols = [col for col in lstm_df.columns if col != target]
        lstm_df = lstm_df[feature_cols + [target]]

        lstm_df = lstm_df.select_dtypes(include=[np.number])

        if target not in lstm_df.columns:
            raise ValueError(f"Target column '{target}' removed during numeric selection")

        split_index = int(len(lstm_df) * 0.8)

        train_lstm_df = lstm_df.iloc[:split_index]
        test_lstm_df = lstm_df.iloc[split_index:]

        lstm_scaler = MinMaxScaler()

        train_scaled = lstm_scaler.fit_transform(train_lstm_df)
        test_scaled = lstm_scaler.transform(test_lstm_df)

        joblib.dump(
            lstm_scaler,
            os.path.join(self.config.root_dir, self.config.lstm_scaler_name)
        )

        X_train_lstm, y_train_lstm = self.create_sequences(
            train_scaled,
            window_size=window_size
        )

        X_test_lstm, y_test_lstm = self.create_sequences(
            test_scaled,
            window_size=window_size
        )

        np.save(os.path.join(self.config.lstm_dir, "X_train.npy"), X_train_lstm)
        np.save(os.path.join(self.config.lstm_dir, "X_test.npy"), X_test_lstm)
        np.save(os.path.join(self.config.lstm_dir, "y_train.npy"), y_train_lstm)
        np.save(os.path.join(self.config.lstm_dir, "y_test.npy"), y_test_lstm)

        print("LSTM data saved successfully.")
        print(f"X_train_lstm shape: {X_train_lstm.shape}")
        print(f"X_test_lstm shape: {X_test_lstm.shape}")
        print(f"y_train_lstm shape: {y_train_lstm.shape}")
        print(f"y_test_lstm shape: {y_test_lstm.shape}")

    def initiate_data_transformation(self):
        try:
            df = pd.read_csv(self.config.data_path)

            print("Original dataframe shape:", df.shape)
            print("Original dtypes:")
            print(df.dtypes)

            df = self.clean_target_column(df)
            df = self.feature_engineering(df)
            df = self.handle_missing_values(df)
            df = self.handle_encoding(df)

            print("Processed dataframe shape:", df.shape)
            print("Processed dtypes:")
            print(df.dtypes)

            classic_df = df.drop(columns=["Date"], errors="ignore")

            train, test = train_test_split(
                classic_df,
                test_size=0.2,
                shuffle=False
            )

            self.save_ensemble_data(train, test)
            print("Ensemble train/test saved successfully.")

            self.save_regression_data(train, test)
            print("Regression train/test saved successfully.")

            self.save_lstm_data(df, window_size=30)

            print("All data transformation steps completed successfully.")

        except Exception as e:
            raise CustomException(e,sys)


In [16]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()

    data_transformation = DataTransformation(
        config=data_transformation_config
    )

    data_transformation.initiate_data_transformation()

except Exception as e:
    raise CustomException(e,sys)


{"path": "config\\config.yaml", "timestamp": "2026-06-24T17:57:07.359404Z", "level": "info", "event": "YAML file loaded successfully"}
{"path": "params.yaml", "timestamp": "2026-06-24T17:57:07.362676Z", "level": "info", "event": "YAML file loaded successfully"}
{"path": "schema.yaml", "timestamp": "2026-06-24T17:57:07.368608Z", "level": "info", "event": "YAML file loaded successfully"}
{"path": "artifacts", "timestamp": "2026-06-24T17:57:07.371053Z", "level": "info", "event": "Created directory"}
{"path": "artifacts\\data_transformation", "timestamp": "2026-06-24T17:57:07.373294Z", "level": "info", "event": "Created directory"}
{"path": "artifacts\\data_transformation\\regression", "timestamp": "2026-06-24T17:57:07.376146Z", "level": "info", "event": "Created directory"}
{"path": "artifacts\\data_transformation\\ensemble", "timestamp": "2026-06-24T17:57:07.378768Z", "level": "info", "event": "Created directory"}
{"path": "artifacts\\data_transformation\\lstm", "timestamp": "2026-06-24T

Original dataframe shape: (169211, 11)
Original dtypes:
Product_id           int64
Product_Code        object
Warehouse           object
Product_Category    object
Date                object
Order_Demand         int64
Open                 int64
Promo                int64
StateHoliday        object
SchoolHoliday        int64
Petrol_price         int64
dtype: object
Processed dataframe shape: (169211, 16)
Processed dtypes:
Product_id                   int64
Product_Code                 int64
Warehouse                    int64
Product_Category             int64
Date                datetime64[ns]
Order_Demand                 int64
Open                         int64
Promo                        int64
StateHoliday                 int64
SchoolHoliday                int64
Petrol_price                 int64
year                         int32
month                        int32
day                          int32
dayofweek                    int32
weekofyear                   int64
dtype: object
E